# DICOM to 3D Surgical Model — Pipeline Demo

**Autor:** Ricardo Garza Benítez · Ingeniería Biomédica, UDEM
**Proyecto:** Pipeline DICOM → STL para modelos anatómicos en simulación quirúrgica

---

## ¿Qué hace este pipeline?

Los escáneres CT producen cientos de imágenes 2D llamadas **archivos DICOM**.
Juntas forman un volumen 3D de datos de radiodensidad medidos en **Unidades Hounsfield (HU)**.
Este notebook convierte ese volumen en un mesh 3D exportable como `.STL`.

```
CT scanner → archivos DICOM → volumen 3D (HU) → segmentación → mesh STL → impresión 3D
```

## Contexto clínico

Los modelos anatómicos físicos permiten a los cirujanos:

- **Planificar** procedimientos complejos antes de entrar al quirófano
- **Practicar** osteotomías y colocación de implantes sobre réplicas del paciente real
- **Comunicar** diagnósticos a pacientes con un modelo tangible de su propia anatomía



---
## Setup — Importar librerías

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Añadir la raíz del proyecto al path para poder importar desde src/
sys.path.insert(0, '..')

from src.loader       import load_dicom_series
from src.preprocessor import apply_window, remove_table, resample_volume
from src.segmentor    import segment_bone
from src.mesh_utils   import (build_mesh, keep_largest_component,
                               smooth_mesh, fix_mesh, export_stl, get_mesh_stats)
from src.visualizer   import show_slices, show_volume_stats, render_mesh_preview

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print("Librerías cargadas correctamente.")

---
## Paso 1 — Cargar la serie DICOM

### ¿Qué es un archivo DICOM?

**DICOM** (Digital Imaging and Communications in Medicine) es el estándar universal
para imágenes médicas. Cada archivo `.dcm` contiene:

- **Datos de píxeles**: la imagen 2D en valores raw del detector
- **Metadatos clínicos**: posición del slice en el espacio 3D, espaciado de píxeles,
  modalidad (CT/MRI), datos del paciente (anonimizados en datasets públicos)

El tag crítico para reconstruir el volumen es `ImagePositionPatient[2]` — la coordenada Z
de cada slice. **Ordenar por este valor** (no por nombre de archivo) garantiza la secuencia
anatómica correcta, ya que los archivos en carpeta no siempre están en orden.

### Unidades Hounsfield (HU)

Los valores raw del detector se convierten a una escala universal con la fórmula:

```
HU = pixel_value × RescaleSlope + RescaleIntercept
```

| Tejido | HU |
|--------|-----|
| Aire | −1000 |
| Grasa | −100 a −50 |
| Agua | 0 |
| Tejido blando | 20 – 80 |
| Hueso esponjoso (trabecular) | 150 – 400 |
| Hueso cortical | 400 – 1000 |
| Metal / implantes | > 1000 |

Esta calibración permite usar el **mismo umbral** para segmentar hueso en cualquier
escáner del mundo — es lo que hace al pipeline reproducible clínicamente.

In [ ]:
DATA_DIR = '../data/sample'

volume, metadata = load_dicom_series(DATA_DIR)

print(f"Volumen cargado:")
print(f"  Shape     : {volume.shape}  → (slices, filas, columnas)")
print(f"  Tipo dato : {volume.dtype}")
print(f"  HU mínimo : {volume.min()} HU  (~-1000 = aire)")
print(f"  HU máximo : {volume.max()} HU  (>400 confirma presencia de hueso)")
print(f"  Modalidad : {metadata['modality']}")
print(f"  Voxel     : {[round(s, 2) for s in metadata['voxel_size_mm']]} mm  (z, y, x)")
print(f"  Paciente  : {metadata['patient_id']}")

---
## Paso 2 — Explorar el volumen

Antes de procesar es fundamental **visualizar** el volumen para verificar que se cargó
correctamente y entender la anatomía presente.

### Tres vistas ortogonales

Un volumen 3D se corta en tres planos perpendiculares:

- **Axial** (transversal): visto desde arriba → el plano que los radiólogos leen habitualmente
- **Coronal**: de frente al paciente → muestra la columna vertebral y las crestas ilíacas
- **Sagital**: de perfil → muestra el arco lumbar y el sacro

### Histograma HU

El histograma revela qué tejidos están presentes. En un CT abdominal/pélvico
de colonografía esperamos peaks en:

- ~−1000 HU: aire en el colon (preparación con insuflación de CO₂)
- ~0 HU: agua y tejidos blandos
- ~400–1000 HU: hueso cortical (columna, pelvis, fémures)

In [ ]:
# Tres vistas ortogonales a través del centro del volumen
show_slices(volume, title='CT Colonography — Vistas ortogonales (HU raw)')

In [ ]:
# Histograma de HU con rangos de tejido anotados + slice axial central
show_volume_stats(volume)

---
## Paso 3 — Preprocesar el volumen

El volumen raw necesita dos correcciones antes de poder segmentar.

### 1 · Eliminar la camilla del escáner

La camilla del CT es radiodensa (HU ~200–400) y podría confundirse con hueso.
Se elimina identificando la **región conectada más grande** por encima de −500 HU
(el cuerpo del paciente) y seteando todo lo demás a −1000 HU (aire).

### 2 · Resampleo isótropo

Los escáneres CT tienen alta resolución en el plano XY (0.6–0.8 mm) pero mayor
espesor de slice en Z (1.25–5 mm). Esto genera voxels "rectangulares" que distorsionan
la geometría del mesh:

```
Antes:  voxel = 1.25 × 0.66 × 0.66 mm  → mesh con artefactos en Z
Después: voxel = 1.00 × 1.00 × 1.00 mm  → geometría correcta
```

### HU Windowing

El windowing restringe el rango de HU visible para resaltar un tejido específico.
La **ventana de hueso** (centro = 400 HU, ancho = 1000 HU) mapea el rango [−100, 900] HU
a escala de grises [0, 1], haciendo que el hueso aparezca brillante y el tejido blando oscuro.

In [ ]:
# Visualizar el efecto del windowing sobre el slice central
mid = volume.shape[0] // 2
vol_windowed = apply_window(volume, window='bone')  # normaliza a [0.0, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Efecto del Windowing de Hueso  (centro=400 HU, ancho=1000 HU)',
             fontsize=13, fontweight='bold')

axes[0].imshow(volume[mid], cmap='gray', vmin=-1000, vmax=3000)
axes[0].set_title(f'Raw HU — rango completo ({volume.min()} a {volume.max()} HU)')
axes[0].axis('off')

axes[1].imshow(vol_windowed[mid], cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Ventana de hueso — hueso=blanco, tejido blando=gris oscuro, aire=negro')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Eliminar camilla: setea píxeles fuera del cuerpo a -1000 HU
vol_clean = remove_table(volume)
print("Camilla eliminada.")

# 2. Resamplear a voxels isótrópicos de 1 mm³
original_spacing = metadata['voxel_size_mm']
vol_resampled, new_spacing = resample_volume(
    vol_clean, original_spacing, target_spacing=[1.0, 1.0, 1.0]
)

print(f"\nResampleo completado:")
print(f"  Shape original  : {vol_clean.shape}")
print(f"  Spacing original: {[round(s, 2) for s in original_spacing]} mm  (z, y, x)")
print(f"  Shape nuevo     : {vol_resampled.shape}")
print(f"  Spacing nuevo   : {[round(s, 2) for s in new_spacing]} mm  (1 mm isótropo)")

---
## Paso 4 — Segmentar con Marching Cubes

### ¿Qué es Marching Cubes?

**Marching Cubes** (Lorensen & Cline, 1987) extrae isosuperficies de campos
volumétricos. Es el algoritmo detrás de prácticamente todo el software de
segmentación médica 3D. Funciona así:

1. Divide el volumen en cubos de 2×2×2 voxels
2. Para cada cubo, evalúa si sus 8 vértices están **por encima o por debajo** del umbral HU
3. Hay 256 combinaciones posibles → solo 15 patrones únicos por simetría
4. Genera triángulos que aproximan la superficie en cada cubo
5. Concatena todos los triángulos → mesh 3D completo

### Umbral de segmentación para hueso cortical

Usamos **400 HU** — el límite entre tejido blando y hueso denso.

- Voxels con HU > 400 → "dentro" del sólido
- Voxels con HU < 400 → "fuera"
- La superficie se extrae exactamente en ese límite

Un suavizado Gaussiano leve (σ=1) antes de Marching Cubes reduce el artefacto
de "escalera" que producen los voxels discretos sobre la superficie resultante.

### Anatomía capturada

En un CT abdominal/pélvico de colonografía, la segmentación de hueso incluye:
vértebras lumbares, sacro, crestas ilíacas, acetábulo, cabezas femorales y costillas inferiores.

In [ ]:
# Segmentar hueso cortical — umbral 400 HU
# El parámetro spacing convierte índices de voxel a coordenadas en milímetros
verts, faces, normals = segment_bone(
    vol_resampled,
    spacing=new_spacing,
    bone_type='cortical'   # 'cortical' = 400 HU  |  'trabecular' = 150 HU
)

print(f"Surface mesh extraído:")
print(f"  Vértices    : {len(verts):,}")
print(f"  Triángulos  : {len(faces):,}")
print(f"  Extensión X : {verts[:, 0].min():.1f} — {verts[:, 0].max():.1f} mm")
print(f"  Extensión Y : {verts[:, 1].min():.1f} — {verts[:, 1].max():.1f} mm")
print(f"  Extensión Z : {verts[:, 2].min():.1f} — {verts[:, 2].max():.1f} mm")
print(f"\nEl mesh raw incluye fragmentos pequeños (ruido de escáner).")
print(f"Los limpiaremos en el siguiente paso.")

---
## Paso 5 — Construir, limpiar y exportar el mesh

### Problemas del mesh raw

Marching Cubes sobre datos CT produce tres artefactos típicos:

**1. Fragmentos desconectados**
Ruido del escáner, artefactos de metal o burbujas de gas producen pequeñas "islas"
flotantes sin relación anatómica. Se eliminan conservando solo el **componente
conectado más grande** (el esqueleto principal).

**2. Efecto escalera (staircase artifact)**
Los voxels son cubos discretos → la superficie tiene escalones visibles.
El **suavizado Laplaciano** mueve cada vértice hacia el promedio de sus vecinos,
suavizando la superficie sin alterar la topología. 3–10 iteraciones es el rango clínico.

**3. Mesh no-manifold**
Bordes compartidos por más de 2 triángulos o normales inconsistentes.
`fill_holes()` + `fix_normals()` corrigen esto para compatibilidad con impresoras 3D.

### Formato STL

STL (STereoLithography) es el formato universal para fabricación aditiva y simulación.
En modo binario: **50 bytes por triángulo** (normal + 3 vértices × 3 coordenadas float32).
Compatible con Mimics, 3D Slicer, Meshmixer, Cura, PrusaSlicer, etc.

In [ ]:
import trimesh

STL_PATH = '../output/stl/bone.stl'

# 1. Crear objeto trimesh desde vértices y caras crudas
mesh = build_mesh(verts, faces)
print(f"[1] Mesh inicial     : {len(mesh.faces):,} caras, {len(mesh.vertices):,} vértices")

# 2. Quedarse con el componente conectado más grande
mesh = keep_largest_component(mesh)
print(f"[2] Tras limpieza    : {len(mesh.faces):,} caras")

# 3. Suavizado Laplaciano (5 iteraciones)
mesh = smooth_mesh(mesh, iterations=5)

# 4. Cerrar huecos y corregir orientación de normales
mesh = fix_mesh(mesh)

# 5. Exportar STL binario
os.makedirs(os.path.dirname(STL_PATH), exist_ok=True)
export_stl(mesh, STL_PATH)

# 6. Stats finales
stats = get_mesh_stats(mesh)
print(f"\n{'='*48}")
print(f"  Mesh final")
print(f"{'='*48}")
print(f"  Vértices    : {stats['n_vertices']:,}")
print(f"  Caras       : {stats['n_faces']:,}")
print(f"  Watertight  : {stats['is_watertight']}")
if stats['volume_cm3']:
    print(f"  Volumen     : {stats['volume_cm3']:.1f} cm³")
print(f"  Superficie  : {stats['surface_area_cm2']:.1f} cm²")
dims = stats['dimensions_mm']
print(f"  Dimensiones : {dims[0]:.0f} × {dims[1]:.0f} × {dims[2]:.0f} mm")

---
## Paso 6 — Visualización 3D del mesh

El render muestra el mesh desde las **cuatro vistas clínicas estándar**,
con iluminación difusa simulada (producto punto de normales de cara con
un vector de luz) que da apariencia de hueso real.

Terminología anatómica de las vistas:

| Vista | Descripción |
|-------|-------------|
| **Anterior** | De frente al paciente (ventral) |
| **Lateral** | Perfil izquierdo o derecho |
| **Superior** | Desde la cabeza hacia los pies (cefálico) |
| **Isométrico** | Perspectiva oblicua — la más usada en presentaciones clínicas |

La segmentación captura la estructura ósea completa visible en el campo de visión
del CT de colonografía: columna lumbar, sacro, crestas ilíacas, fémures proximales
y costillas inferiores.

In [ ]:
# Mostrar el render 3D generado por render_mesh_preview()
PREVIEW_PATH = '../output/renders/mesh_preview.png'

img = mpimg.imread(PREVIEW_PATH)
fig, ax = plt.subplots(figsize=(18, 5))
ax.imshow(img)
ax.axis('off')
ax.set_title(
    'Segmentación ósea — CT Colonography (Marching Cubes + Laplacian smoothing)',
    fontsize=13, pad=12
)
plt.tight_layout()
plt.show()

In [ ]:
# Vistas ortogonales del volumen preprocesado (guardadas durante el pipeline)
SLICES_PATH = '../output/renders/slices.png'

img2 = mpimg.imread(SLICES_PATH)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(img2)
ax.axis('off')
ax.set_title('Vistas ortogonales del volumen tras preprocesado', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

---
## Resumen del pipeline

| Paso | Módulo | Operación | Parámetro clave |
|------|--------|-----------|-----------------|
| 1 | `loader.py` | Lee y ordena `.dcm` → HU | `ImagePositionPatient[2]` |
| 2 | `preprocessor.py` | Elimina camilla del escáner | Umbral −500 HU |
| 2 | `preprocessor.py` | Resampleo isótropo | 1.0 × 1.0 × 1.0 mm |
| 3 | `segmentor.py` | Marching Cubes | 400 HU (cortical) |
| 4 | `mesh_utils.py` | Componente más grande + smoothing | 5 iter. Laplaciano |
| 5 | `mesh_utils.py` | Exporta STL binario | 50 bytes / triángulo |

## Próximos pasos

- **Segmentación por región de interés**: crop al fémur o pelvis → mesh watertight
- **Redes neuronales**: reemplazar umbral HU fijo por una CNN de segmentación
  (nnU-Net) para estructuras complejas como cartílago o vasos
- **Simulación quirúrgica**: importar el STL a software FEA (FEBio, Abaqus)
  para simular fuerzas de implante

---
*Ricardo Garza Benítez · Biomedical Engineering,
 2026*